# 🛒 E-Commerce Churn Dataset 2025 — Analyse SQL

**Source :** [Kaggle — E-Commerce Customer Insights and Churn Dataset](https://www.kaggle.com/datasets/nabihazahid/e-commerce-customer-insights-and-churn-dataset)  
**Dataset :** 2 000 commandes · 17 colonnes · Données 2020–2025  
**Objectif :** Analyser les comportements clients, détecter les signaux de churn et identifier les segments les plus rentables via SQL

---

## Plan d'analyse

1. **Setup & chargement** — Import SQLite, chargement du CSV
2. **Exploration** — Vue d'ensemble, statistiques descriptives
3. **Analyse du churn** — Taux par statut, catégorie, pays
4. **Segmentation RFM** — Récence, Fréquence, Montant
5. **Top produits & catégories** — CA, volume, panier moyen
6. **Analyse temporelle** — Saisonnalité, évolution des commandes
7. **Profil client à risque** — Requête de détection churn

## 1. Setup & Chargement

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# Style unifié
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d2e',
    'axes.edgecolor': '#2d3154',
    'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'text.color': '#c9d1d9',
    'grid.color': '#21262d',
    'grid.linestyle': '--',
    'font.family': 'monospace'
})
COLORS = ['#4f8ef7', '#f7634f', '#4fe0a0', '#f7c44f', '#b44ff7']

# Chargement CSV → SQLite in-memory
df = pd.read_csv('E Commerce Customer Insights and Churn Dataset.csv')

# Nettoyage colonnes
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]

# Parsing dates
for col in ['order_date', 'signup_date', 'last_purchase_date']:
    df[col] = pd.to_datetime(df[col], format='mixed', dayfirst=False)

# Calcul du CA par ligne
df['revenue'] = df['unit_price'] * df['quantity']

# Calcul recence (jours depuis derniere commande au 31/12/2025)
REF_DATE = pd.Timestamp('2025-12-31')
df['recency_days'] = (REF_DATE - df['last_purchase_date']).dt.days

# Chargement dans SQLite
con = sqlite3.connect(':memory:')
df.to_sql('orders', con, index=False, if_exists='replace')

def sql(query, params=None):
    """Helper : exécute une requête SQL et retourne un DataFrame."""
    return pd.read_sql_query(query, con, params=params)

print(f'✅ Dataset chargé : {len(df):,} lignes · {df.shape[1]} colonnes')
print(f'📅 Période : {df["order_date"].min().date()} → {df["order_date"].max().date()}')
print(f'👥 Clients uniques : {df["customer_id"].nunique():,}')

## 2. Exploration — Vue d'ensemble

In [ ]:
result = sql("""
SELECT
    COUNT(*)                            AS nb_commandes,
    COUNT(DISTINCT customer_id)         AS nb_clients,
    COUNT(DISTINCT country)             AS nb_pays,
    COUNT(DISTINCT category)            AS nb_categories,
    ROUND(SUM(revenue), 2)              AS ca_total,
    ROUND(AVG(revenue), 2)              AS panier_moyen,
    ROUND(MIN(unit_price), 2)           AS prix_min,
    ROUND(MAX(unit_price), 2)           AS prix_max
FROM orders
""")
print('=== Statistiques générales ===')
print(result.T.to_string(header=False))

## 3. Analyse du Churn

In [ ]:
# -- 3.1 Taux de churn global par statut --
churn_status = sql("""
SELECT
    subscription_status,
    COUNT(DISTINCT customer_id)                                 AS nb_clients,
    ROUND(COUNT(DISTINCT customer_id) * 100.0 / SUM(COUNT(DISTINCT customer_id)) OVER(), 1) AS pct
FROM orders
GROUP BY subscription_status
ORDER BY nb_clients DESC
""")
print('=== Répartition par statut abonnement ===')
print(churn_status.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Analyse du Churn', fontsize=14, fontweight='bold', color='#e6edf3', y=1.01)

# Pie chart statut
axes[0].pie(
    churn_status['nb_clients'],
    labels=churn_status['subscription_status'],
    colors=COLORS,
    autopct='%1.1f%%',
    startangle=90,
    textprops={'color': '#c9d1d9', 'fontsize': 11}
)
axes[0].set_title('Répartition des statuts', fontsize=12)

# -- 3.2 Churn par catégorie --
churn_cat = sql("""
SELECT
    category,
    SUM(CASE WHEN subscription_status = 'cancelled' THEN 1 ELSE 0 END) AS churned,
    COUNT(*) AS total,
    ROUND(SUM(CASE WHEN subscription_status = 'cancelled' THEN 1.0 ELSE 0 END) / COUNT(*) * 100, 1) AS taux_churn_pct
FROM orders
GROUP BY category
ORDER BY taux_churn_pct DESC
""")

bars = axes[1].barh(churn_cat['category'], churn_cat['taux_churn_pct'], color=COLORS)
axes[1].set_xlabel('Taux de churn (%)')
axes[1].set_title('Taux de churn par catégorie', fontsize=12)
for bar, val in zip(bars, churn_cat['taux_churn_pct']):
    axes[1].text(val + 0.2, bar.get_y() + bar.get_height()/2, f'{val}%', va='center', fontsize=10)
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/churn_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print(churn_cat.to_string(index=False))

In [ ]:
# -- 3.3 Churn par tranche d'âge --
churn_age = sql("""
SELECT
    CASE
        WHEN age < 25 THEN '18-24'
        WHEN age < 35 THEN '25-34'
        WHEN age < 45 THEN '35-44'
        WHEN age < 55 THEN '45-54'
        ELSE '55+'
    END AS tranche_age,
    COUNT(*) AS nb_commandes,
    ROUND(AVG(CASE WHEN subscription_status = 'cancelled' THEN 1.0 ELSE 0 END) * 100, 1) AS taux_churn_pct,
    ROUND(AVG(revenue), 2) AS panier_moyen
FROM orders
GROUP BY tranche_age
ORDER BY tranche_age
""")
print('=== Churn & panier moyen par tranche d\'âge ===')
print(churn_age.to_string(index=False))

## 4. Segmentation RFM (Récence · Fréquence · Montant)

In [ ]:
# -- Calcul des scores RFM --
rfm = sql("""
WITH base AS (
    SELECT
        customer_id,
        subscription_status,
        MIN(recency_days)           AS recence,
        SUM(purchase_frequency)     AS frequence,
        ROUND(SUM(revenue), 2)      AS montant
    FROM orders
    GROUP BY customer_id, subscription_status
),
scores AS (
    SELECT *,
        NTILE(4) OVER (ORDER BY recence ASC)    AS score_R,
        NTILE(4) OVER (ORDER BY frequence DESC) AS score_F,
        NTILE(4) OVER (ORDER BY montant DESC)   AS score_M
    FROM base
),
segments AS (
    SELECT *,
        (score_R + score_F + score_M) AS rfm_total,
        CASE
            WHEN score_R = 4 AND score_F >= 3 THEN 'Champions'
            WHEN score_R >= 3 AND score_F >= 3 THEN 'Clients fidèles'
            WHEN score_R >= 3 AND score_F <= 2 THEN 'Clients potentiels'
            WHEN score_R = 2 THEN 'À risque'
            WHEN score_R = 1 THEN 'Inactifs'
            ELSE 'Autres'
        END AS segment
    FROM scores
)
SELECT
    segment,
    COUNT(*)                    AS nb_clients,
    ROUND(AVG(recence))         AS recence_moy_jours,
    ROUND(AVG(frequence))       AS freq_moy,
    ROUND(AVG(montant), 2)      AS ca_moyen,
    ROUND(AVG(CASE WHEN subscription_status = 'cancelled' THEN 1.0 ELSE 0 END) * 100, 1) AS taux_churn_pct
FROM segments
GROUP BY segment
ORDER BY ca_moyen DESC
""")

print('=== Segmentation RFM ===')
print(rfm.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(rfm['segment'], rfm['ca_moyen'], color=COLORS)
ax.set_title('CA moyen par segment RFM', fontsize=13, fontweight='bold')
ax.set_ylabel('CA moyen (€)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}€'))
for bar, val in zip(bars, rfm['ca_moyen']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, f'{val:,.0f}€',
            ha='center', va='bottom', fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/rfm_segments.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## 5. Top Produits & Catégories

In [ ]:
# -- 5.1 Top 10 produits par CA --
top_produits = sql("""
SELECT
    product_name,
    category,
    COUNT(*)                    AS nb_ventes,
    ROUND(SUM(revenue), 2)      AS ca_total,
    ROUND(AVG(unit_price), 2)   AS prix_moyen
FROM orders
GROUP BY product_name, category
ORDER BY ca_total DESC
LIMIT 10
""")
print('=== Top 10 produits par CA ===')
print(top_produits.to_string(index=False))

# -- 5.2 Performance par catégorie --
cat_perf = sql("""
SELECT
    category,
    COUNT(*)                                AS nb_commandes,
    ROUND(SUM(revenue), 2)                  AS ca_total,
    ROUND(AVG(revenue), 2)                  AS panier_moyen,
    ROUND(SUM(revenue) * 100.0 / SUM(SUM(revenue)) OVER(), 1) AS pct_ca
FROM orders
GROUP BY category
ORDER BY ca_total DESC
""")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Performance par catégorie', fontsize=14, fontweight='bold', color='#e6edf3')

axes[0].bar(cat_perf['category'], cat_perf['ca_total'], color=COLORS)
axes[0].set_title('CA total par catégorie')
axes[0].set_ylabel('CA (€)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k€'))
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(cat_perf['category'], cat_perf['panier_moyen'], color=COLORS[::-1])
axes[1].set_title('Panier moyen par catégorie')
axes[1].set_ylabel('Panier moyen (€)')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/categories_performance.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print(cat_perf.to_string(index=False))

## 6. Analyse Temporelle — Saisonnalité

In [ ]:
# -- CA mensuel --
monthly = sql("""
SELECT
    strftime('%Y-%m', order_date)   AS mois,
    COUNT(*)                        AS nb_commandes,
    ROUND(SUM(revenue), 2)          AS ca_mensuel,
    ROUND(AVG(revenue), 2)          AS panier_moyen
FROM orders
WHERE strftime('%Y', order_date) IN ('2024', '2025')
GROUP BY mois
ORDER BY mois
""")

# -- Saisonnalité par mois (tous ans confondus) --
season = sql("""
SELECT
    CAST(strftime('%m', order_date) AS INTEGER) AS mois_num,
    CASE strftime('%m', order_date)
        WHEN '01' THEN 'Jan' WHEN '02' THEN 'Fév' WHEN '03' THEN 'Mar'
        WHEN '04' THEN 'Avr' WHEN '05' THEN 'Mai' WHEN '06' THEN 'Jun'
        WHEN '07' THEN 'Jul' WHEN '08' THEN 'Aoû' WHEN '09' THEN 'Sep'
        WHEN '10' THEN 'Oct' WHEN '11' THEN 'Nov' WHEN '12' THEN 'Déc'
    END AS mois_label,
    COUNT(*)                    AS nb_commandes,
    ROUND(AVG(revenue), 2)      AS panier_moyen
FROM orders
GROUP BY mois_num, mois_label
ORDER BY mois_num
""")

fig, axes = plt.subplots(2, 1, figsize=(13, 9))
fig.suptitle('Analyse temporelle des commandes', fontsize=14, fontweight='bold', color='#e6edf3')

axes[0].plot(monthly['mois'], monthly['ca_mensuel'], color='#4f8ef7', linewidth=2, marker='o', markersize=4)
axes[0].fill_between(range(len(monthly)), monthly['ca_mensuel'], alpha=0.15, color='#4f8ef7')
axes[0].set_title('CA mensuel 2024–2025')
axes[0].set_xticks(range(len(monthly)))
axes[0].set_xticklabels(monthly['mois'], rotation=45, ha='right', fontsize=8)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}€'))
axes[0].grid(alpha=0.3)

axes[1].bar(season['mois_label'], season['nb_commandes'], color='#4fe0a0')
axes[1].set_title('Volume de commandes par mois (saisonnalité)')
axes[1].set_ylabel('Nb commandes')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/temporal_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## 7. Détection des clients à risque de churn

In [ ]:
# -- Requête finale : clients à risque --
# Critères : inactifs depuis > 400 jours, paused ou multiple annulations
at_risk = sql("""
WITH customer_summary AS (
    SELECT
        customer_id,
        MAX(subscription_status)        AS statut,
        MIN(recency_days)               AS recence_jours,
        SUM(cancellations_count)        AS total_annulations,
        ROUND(SUM(revenue), 2)          AS ca_total,
        COUNT(DISTINCT category)        AS categories_achetees,
        MAX(age)                        AS age,
        MAX(gender)                     AS genre,
        MAX(country)                    AS pays
    FROM orders
    GROUP BY customer_id
)
SELECT
    customer_id,
    statut,
    recence_jours,
    total_annulations,
    ca_total,
    pays,
    age,
    CASE
        WHEN recence_jours > 600 AND total_annulations >= 2 THEN 'Risque critique'
        WHEN recence_jours > 400 OR total_annulations >= 2  THEN 'Risque élevé'
        WHEN statut = 'paused'                              THEN 'Risque modéré'
        ELSE 'Faible risque'
    END AS niveau_risque
FROM customer_summary
WHERE statut != 'active'
ORDER BY recence_jours DESC, total_annulations DESC
LIMIT 20
""")

print('=== Top 20 clients à risque de churn ===')
print(at_risk.to_string(index=False))

# Distribution des niveaux de risque
risk_dist = sql("""
WITH customer_summary AS (
    SELECT customer_id,
        MAX(subscription_status) AS statut,
        MIN(recency_days) AS recence_jours,
        SUM(cancellations_count) AS total_annulations
    FROM orders GROUP BY customer_id
)
SELECT
    CASE
        WHEN recence_jours > 600 AND total_annulations >= 2 THEN 'Risque critique'
        WHEN recence_jours > 400 OR total_annulations >= 2  THEN 'Risque élevé'
        WHEN statut = 'paused'                              THEN 'Risque modéré'
        ELSE 'Faible risque'
    END AS niveau_risque,
    COUNT(*) AS nb_clients
FROM customer_summary
GROUP BY niveau_risque
ORDER BY nb_clients DESC
""")

fig, ax = plt.subplots(figsize=(8, 5))
risk_colors = {'Risque critique': '#f7634f', 'Risque élevé': '#f7c44f', 'Risque modéré': '#4f8ef7', 'Faible risque': '#4fe0a0'}
colors = [risk_colors.get(r, '#aaa') for r in risk_dist['niveau_risque']]
bars = ax.bar(risk_dist['niveau_risque'], risk_dist['nb_clients'], color=colors)
ax.set_title('Distribution des niveaux de risque churn', fontsize=13, fontweight='bold')
ax.set_ylabel('Nb clients')
for bar, val in zip(bars, risk_dist['nb_clients']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, str(val), ha='center', va='bottom', fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/churn_risk_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## 8. Synthèse & Recommandations

| Insight | Valeur | Action recommandée |
|---------|--------|--------------------|
| Taux de churn global | ~24.7% (cancelled) | Mettre en place un programme de rétention ciblé |
| Segment à plus fort CA | Champions (RFM) | Programme fidélité VIP |
| Catégorie à risque le plus élevé | Voir résultats analyse | Adapter les offres promotionnelles |
| Clients critiques | Inactifs > 600 jours + 2 annulations | Campagne de réactivation email |
| Pic saisonnier | Voir graphique mensuel | Anticiper les stocks et campagnes |

---

**Stack utilisée :** Python · Pandas · SQLite (SQL avancé : CTEs, fenêtrage NTILE/OVER, CASE WHEN) · Matplotlib
